# 🚗 VLA Simulation — Colab 원클릭 실습 (무료 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKKUAutoLab/VLA_Simulation/blob/main/colab/VLA_Simulation_Colab.ipynb)

### 딱 2단계면 끝
1. 위 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 선택 → 저장
2. **런타임 → 모두 실행** (또는 `Ctrl+F9`)

그러면 clone → ROS2/Gazebo 설치 → 모델 import → 빌드 → 주행까지 자동 진행됩니다.
Gazebo 화면은 아래 셀이 출력하는 **noVNC 링크**를 클릭해 브라우저에서 봅니다.

> ⚠️ ROS2/Gazebo 설치(셀 2)는 수 분 걸립니다. 그대로 기다리면 됩니다.

## 0. GPU 확인

In [ ]:
# GPU 확인 — 없으면 아래 안내대로 런타임을 GPU로 바꾸세요
import torch, subprocess
has_gpu = torch.cuda.is_available()
print('CUDA 사용 가능:', has_gpu)
if has_gpu:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout.split('\n')[8:11])
else:
    print('\n' + '='*60)
    print('⚠️  GPU가 꺼져 있습니다. VLA 추론에는 GPU가 필요합니다.')
    print('   런타임 → 런타임 유형 변경 → T4 GPU 선택 후, 런타임 → 모두 실행')
    print('='*60)

## 1. 저장소 클론

In [ ]:
%cd /content
# repo가 Public 이어야 익명 clone 가능. private면 여기서 실패합니다.
!git clone https://github.com/SKKUAutoLab/VLA_Simulation.git ros2_autonomous_vehicle_simulation
import os, sys
if not os.path.isdir('/content/ros2_autonomous_vehicle_simulation/lora_pipeline'):
    raise SystemExit('❌ clone 실패: repo가 Public 인지, 파일이 기본 브랜치(main)에 있는지 확인하세요.')
%cd /content/ros2_autonomous_vehicle_simulation
print('✅ clone OK')

## 2. ROS 2 Humble + Gazebo Classic 설치 (Colab = Ubuntu 22.04)

In [ ]:
# ROS2 apt 소스 추가 후 ros-humble-desktop + gazebo 설치 (수 분 소요)
!apt-get update -q && apt-get install -y -q software-properties-common curl gnupg lsb-release
!curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg
!echo "deb [signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu jammy main" > /etc/apt/sources.list.d/ros2.list
!apt-get update -q && apt-get install -y -q ros-humble-ros-base ros-humble-gazebo-ros-pkgs gazebo
!pip install -q transformers peft accelerate huggingface_hub

## 3. 사전 학습 산출물 — HuggingFace Hub에서 import (데이터·학습 생략)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download
REPO = 'hoonsy/VLA_Simulation-pretrained'   # 사전 학습 산출물 배포 repo
DEST = '/content/ros2_autonomous_vehicle_simulation/lora_pipeline'
assert os.path.isdir('/content/ros2_autonomous_vehicle_simulation'), '먼저 셀 1(clone)이 성공해야 합니다.'
os.makedirs(DEST, exist_ok=True)   # 폴더 없으면 생성 (안전)
for f in ('vla_lora_head_fast.pt', 'vla_lora_head.pt'):
    shutil.copy(hf_hub_download(repo_id=REPO, filename=f), f'{DEST}/{f}')
d = snapshot_download(repo_id=REPO, allow_patterns='vla_lora_adapter/*')
shutil.copytree(f'{d}/vla_lora_adapter', f'{DEST}/vla_lora_adapter', dirs_exist_ok=True)
print('✅ 산출물 로드 완료 — 데이터 수집·학습 없이 추론 가능')

## 4. Gazebo를 브라우저에서 보기 — Xvfb + noVNC
실행 후 출력되는 noVNC 링크를 클릭하면 브라우저에서 Gazebo 3D 화면을 볼 수 있습니다.

In [ ]:
!apt-get install -y -q xvfb x11vnc websockify novnc >/dev/null
import os, subprocess, time
os.environ['DISPLAY'] = ':1'
subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1280x800x24'])
time.sleep(2)
subprocess.Popen(['x11vnc', '-display', ':1', '-nopw', '-forever', '-shared', '-rfbport', '5900'])
subprocess.Popen(['websockify', '--web=/usr/share/novnc', '6080', 'localhost:5900'])
from google.colab.output import eval_js
print('noVNC:', eval_js('google.colab.kernel.proxyPort(6080)') + 'vnc.html')

## 5. 빌드 & VLA 주행 실행 (Xvfb 디스플레이 → noVNC로 관찰)

In [ ]:
%%bash
source /opt/ros/humble/setup.bash
cd /content/ros2_autonomous_vehicle_simulation
colcon build --packages-select interfaces_pkg --allow-overriding interfaces_pkg && source install/local_setup.bash
colcon build --symlink-install --packages-select simulation_pkg camera_perception_pkg lidar_perception_pkg qwen_vl_pkg
source install/local_setup.bash
# 주행 (noVNC 탭에서 Gazebo 확인)
DISPLAY=:1 ros2 launch lora_pipeline/vla_drive.launch.py gzclient:=true